In [1]:
import sys
sys.path.insert(0, '/home/joseluisalmendarezgonzalez/Desktop/3_GEO_FNO')

In [5]:
import torch
import random
import numpy as np
from torch.utils.data import DataLoader, random_split
from lib.Common import setup_logging, KolmogorovDataset, FNOGenerator, NavierStokesResiduo
from lib.PurePhysLossApproach import FNOPhysicsTrainer

In [6]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [7]:
setup_logging("physics_experiment.log")

In [10]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_PATH = "../Dataset/snapshots_64x64_use.npy"

In [11]:
dataset = KolmogorovDataset(DATA_PATH, seq_len=10)
train_len = int(0.8 * len(dataset))
val_len = len(dataset) - train_len
train_ds, val_ds = random_split(dataset, [train_len, val_len])

2026-06-07 19:40:09,378 | INFO | Dataset: 1152 trayectorias × 90 ventanas = 103,680 muestras | seq_len=10 | H×W=64×64


In [12]:
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=4)

In [13]:
G = FNOGenerator(hidden_ch=32, modes1=12, modes2=12, n_layers=4, z_dim=0).to(DEVICE)
ns = NavierStokesResiduo(64, 64, dt=0.05, modes1=12, modes2=12, device=DEVICE).to(DEVICE)

In [14]:
trainer = FNOPhysicsTrainer(
    G, ns, DEVICE,
    lr=1e-4,
    log_dir="logs_physics",
    resume=True,
)

2026-06-07 19:40:11,991 | WARNING | No se encontró checkpoint para reanudar; empezando desde cero.


In [15]:
history = trainer.fit(train_loader, val_loader, epochs=30)
print(history)

Training Physics:   0%|                                                  | 0/10368 [00:00<?, ?it/s]/home/joseluisalmendarezgonzalez/miniconda3/envs/py_env/lib/python3.10/site-packages/torch/nn/modules/conv.py:456: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,
Training Physics:   0%|                           | 21/10368 [00:05<47:21,  3.64it/s, loss=1.10506]


KeyboardInterrupt: 